# MVP06 — Camada Silver

Transforma as tabelas da Bronze em tabelas limpas, tipadas e padronizadas em `mvp.silver`.
Cada regra aplicada aqui responde a um achado medido no **MVP05-qualidade**. A coluna "Achado" referencia
a linha correspondente da tabela-síntese daquele notebook.

| Tabela Silver | Origem Bronze | Regras aplicadas | Achado MVP05 |
|---|---|---|---|
| `pedidos` | `orders` | Tipagem de datas; status padronizado; `flag_entrega_valida` isola entregas nulas, incoerentes ou de pedidos não entregues | #1, #2 |
| `itens_pedido` | `order_items` | Preço e frete em DECIMAL; sequencial em INT | Consistência de tipos |
| `pagamentos` | `order_payments` | Tipagem; parcelas menores que 1 convertidas para 1; `not_defined` mantido | #10, #11 |
| `avaliacoes` | `order_reviews` | Uma avaliação por pedido (a mais recente); nota fora de 1–5 vira nulo; comentários vazios viram nulo | #4, #5 |
| `clientes` | `customers` | Cidade padronizada; UF em maiúsculas; CEP com 5 dígitos | #9 |
| `vendedores` | `sellers` | Mesmas regras de clientes | #9 |
| `produtos` | `products` + `category_translation` | Categoria nula vira `sem_categoria`; tradução com fallback; medidas ≤ 0 viram nulo; correção dos nomes de coluna com erro de digitação na origem (`lenght`) | #6, #7, #8 |

**Não entram na Silver:** `geolocation` e `category_translation` (incorporada a `produtos`).

**Outliers de preço (#14)** são mantidos: representam produtos legitimamente caros, não erros.

## Configuração e funções auxiliares

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ORIGEM = "mvp.bronce"
DESTINO = "mvp.silver"

ACENTOS    = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuuc"


def bronze(tabela):
    return spark.table(f"{ORIGEM}.{tabela}")


def cast_seguro(coluna, tipo):
    """Converte com try_cast: valores inválidos viram nulo em vez de gerar erro."""
    return F.expr(f"try_cast({coluna} AS {tipo})")


def texto_limpo(coluna):
    """Remove espaços nas bordas; string vazia vira nulo."""
    c = F.trim(F.col(coluna))
    return F.when(c == "", None).otherwise(c)


def padronizar_cidade(coluna):
    """Minúsculas, sem acentos, apenas letras/espaço/hífen/apóstrofo, espaços únicos, capitalizada."""
    c = F.translate(F.lower(F.trim(F.col(coluna))), ACENTOS, SEM_ACENTO)
    c = F.regexp_replace(c, r"[^a-z' -]", "")
    c = F.regexp_replace(c, r"\s+", " ")
    return F.initcap(F.trim(c))


def medida_positiva(coluna):
    """Converte para INT; zero ou negativo vira nulo (medida física impossível)."""
    v = cast_seguro(coluna, "INT")
    return F.when(v > 0, v)


def salvar(df, tabela, comentario):
    df = df.withColumn("_data_processamento", F.current_timestamp())
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{DESTINO}.{tabela}"))
    spark.sql(f"COMMENT ON TABLE {DESTINO}.{tabela} IS '{comentario}'")
    n = spark.table(f"{DESTINO}.{tabela}").count()
    print(f"{DESTINO}.{tabela}: {n:,} linhas")

## silver.pedidos

- Datas convertidas de texto para TIMESTAMP.
- `flag_entrega_valida` = verdadeiro apenas quando o pedido tem status `delivered`, possui data de entrega e essa data não é anterior à compra. Pedidos que falham nesse critério **permanecem na tabela**, mas ficam fora das métricas de prazo e atraso na Gold.

In [0]:
pedidos = (bronze("orders")
    .select(
        F.trim("order_id").alias("order_id"),
        F.trim("customer_id").alias("customer_id"),
        F.lower(F.trim("order_status")).alias("status_pedido"),
        cast_seguro("order_purchase_timestamp", "TIMESTAMP").alias("data_hora_compra"),
        cast_seguro("order_approved_at", "TIMESTAMP").alias("data_aprovacao"),
        cast_seguro("order_delivered_carrier_date", "TIMESTAMP").alias("data_envio"),
        cast_seguro("order_delivered_customer_date", "TIMESTAMP").alias("data_entrega"),
        cast_seguro("order_estimated_delivery_date", "TIMESTAMP").alias("data_estimada"))
    .dropDuplicates(["order_id"])
    .withColumn("flag_entrega_valida",
        F.coalesce(
            (F.col("status_pedido") == "delivered")
            & F.col("data_entrega").isNotNull()
            & (F.col("data_entrega") >= F.col("data_hora_compra")),
            F.lit(False))))

salvar(pedidos, "pedidos",
       "Silver. Pedidos tipados. Grão: um pedido. flag_entrega_valida isola entregas utilizáveis nas métricas de prazo.")

## silver.itens_pedido

In [0]:
itens = (bronze("order_items")
    .select(
        F.trim("order_id").alias("order_id"),
        cast_seguro("order_item_id", "INT").alias("order_item_id"),
        F.trim("product_id").alias("product_id"),
        F.trim("seller_id").alias("seller_id"),
        cast_seguro("shipping_limit_date", "TIMESTAMP").alias("data_limite_envio"),
        cast_seguro("price", "DECIMAL(10,2)").alias("preco"),
        cast_seguro("freight_value", "DECIMAL(10,2)").alias("valor_frete"))
    .dropDuplicates(["order_id", "order_item_id"]))

salvar(itens, "itens_pedido",
       "Silver. Itens de pedido tipados. Grão: um item de pedido. Valores em BRL.")

## silver.pagamentos

- Parcelas menores que 1 são convertidas para 1: todo pagamento tem ao menos uma parcela.
- `payment_type = not_defined` é mantido para não distorcer o valor pago; na Gold ele não é escolhido como meio de pagamento principal.

In [0]:
pagamentos = (bronze("order_payments")
    .select(
        F.trim("order_id").alias("order_id"),
        cast_seguro("payment_sequential", "INT").alias("sequencial_pagamento"),
        F.lower(F.trim("payment_type")).alias("tipo_pagamento"),
        cast_seguro("payment_installments", "INT").alias("parcelas"),
        cast_seguro("payment_value", "DECIMAL(10,2)").alias("valor_pagamento"))
    .withColumn("parcelas",
        F.when(F.col("parcelas") < 1, F.lit(1)).otherwise(F.col("parcelas")))
    .dropDuplicates(["order_id", "sequencial_pagamento"]))

salvar(pagamentos, "pagamentos",
       "Silver. Transações de pagamento tipadas. Grão: uma transação. Parcelas menores que 1 convertidas para 1. Origem: bronce.order_payments.")

## Avaliações

- **Deduplicação :** quando um pedido tem mais de uma avaliação, mantém-se a mais recente (maior data de criação; empate resolvido pela data de resposta). Justificativa: a avaliação mais recente reflete a percepção final do cliente sobre a entrega.
- Nota fora do intervalo 1–5 vira nulo.
- Título e comentário vazios viram nulo (achado #5: campos opcionais, ausência esperada).
- Observação: um mesmo `review_id` pode continuar associado a pedidos diferentes. Isso é aceitável porque o grão desta tabela é **uma avaliação por pedido**, e não um `review_id` único.

In [0]:
janela_recente = Window.partitionBy("order_id").orderBy(
    F.col("data_criacao").desc_nulls_last(),
    F.col("data_resposta").desc_nulls_last())

avaliacoes = (bronze("order_reviews")
    .select(
        F.trim("review_id").alias("review_id"),
        F.trim("order_id").alias("order_id"),
        cast_seguro("review_score", "INT").alias("nota"),
        texto_limpo("review_comment_title").alias("titulo_comentario"),
        texto_limpo("review_comment_message").alias("comentario"),
        cast_seguro("review_creation_date", "TIMESTAMP").alias("data_criacao"),
        cast_seguro("review_answer_timestamp", "TIMESTAMP").alias("data_resposta"))
    .withColumn("nota", F.when(F.col("nota").between(1, 5), F.col("nota")))
    .withColumn("_rn", F.row_number().over(janela_recente))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

salvar(avaliacoes, "avaliacoes",
       "Silver. Avaliações deduplicadas. Grão: uma avaliação por pedido (a mais recente).")

## silver.clientes e silver.vendedores

- Cidade padronizada (achado #9): minúsculas, sem acentos, sem caracteres especiais, espaços únicos, capitalizada. A normalização é intencionalmente simples; grafias muito divergentes podem permanecer. Como as perguntas de negócio usam UF (e não cidade), isso não afeta as respostas.
- UF em Maiúsculas e sem espaços.
- Prefixo de CEP completado com zeros à esquerda até 5 dígitos (preserva CEPs de SP que começam com 0).

In [0]:
clientes = (bronze("customers")
    .select(
        F.trim("customer_id").alias("customer_id"),
        F.trim("customer_unique_id").alias("customer_unique_id"),
        F.lpad(F.trim("customer_zip_code_prefix"), 5, "0").alias("cep_prefixo"),
        padronizar_cidade("customer_city").alias("cidade"),
        F.upper(F.trim("customer_state")).alias("uf"))
    .dropDuplicates(["customer_id"]))

salvar(clientes, "clientes",
       "Silver. Clientes padronizados. Grão: um customer_id (um por pedido). Origem: bronce.customers.")

vendedores = (bronze("sellers")
    .select(
        F.trim("seller_id").alias("seller_id"),
        F.lpad(F.trim("seller_zip_code_prefix"), 5, "0").alias("cep_prefixo"),
        padronizar_cidade("seller_city").alias("cidade"),
        F.upper(F.trim("seller_state")).alias("uf"))
    .dropDuplicates(["seller_id"]))

salvar(vendedores, "vendedores",
       "Silver. Vendedores padronizados. Grão: um vendedor.")

## silver.produtos

- Categoria nula vira `sem_categoria`, para que esses produtos não desapareçam das análises por categoria.
- Tradução via join com `category_translation`; categorias sem tradução recebem o próprio nome em português.
- Peso e dimensões iguais a zero ou negativos viram nulo.
- Colunas renomeadas: a origem traz `product_name_lenght` e `product_description_lenght` (erro de digitação de *length*).
- Proteção contra BOM: o cabeçalho do arquivo de tradução pode trazer o caractere invisível `\ufeff` no primeiro nome de coluna; ele é removido antes do join.

In [0]:
traducao = bronze("category_translation")
traducao = traducao.toDF(*[c.replace("\ufeff", "").strip() for c in traducao.columns])
traducao = traducao.select(
    F.trim("product_category_name").alias("categoria"),
    F.trim("product_category_name_english").alias("categoria_en_origem"))

produtos = (bronze("products")
    .select(
        F.trim("product_id").alias("product_id"),
        F.coalesce(texto_limpo("product_category_name"), F.lit("sem_categoria")).alias("categoria"),
        cast_seguro("product_name_lenght", "INT").alias("qtd_caracteres_nome"),
        cast_seguro("product_description_lenght", "INT").alias("qtd_caracteres_descricao"),
        cast_seguro("product_photos_qty", "INT").alias("qtd_fotos"),
        medida_positiva("product_weight_g").alias("peso_g"),
        medida_positiva("product_length_cm").alias("comprimento_cm"),
        medida_positiva("product_height_cm").alias("altura_cm"),
        medida_positiva("product_width_cm").alias("largura_cm"))
    .dropDuplicates(["product_id"])
    .join(traducao, on="categoria", how="left")
    .withColumn("categoria_en", F.coalesce(F.col("categoria_en_origem"), F.col("categoria")))
    .drop("categoria_en_origem"))

salvar(produtos, "produtos",
       "Silver. Produtos padronizados com categoria traduzida. Grão: um produto. Origem: bronce.products + bronce.category_translation.")

## Validação 1 — Conservação de linhas (Bronze × Silver)

Esperado: diferença zero em todas as tabelas, **exceto `avaliacoes`**, onde a diferença corresponde exatamente às avaliações duplicadas por pedido removidas pela regra de negócio.

In [0]:
import pandas as pd

PARES = [("orders", "pedidos"), ("order_items", "itens_pedido"), ("order_payments", "pagamentos"),
         ("order_reviews", "avaliacoes"), ("customers", "clientes"), ("sellers", "vendedores"),
         ("products", "produtos")]

conservacao = pd.DataFrame([
    {"bronze": b, "silver": s,
     "linhas_bronze": bronze(b).count(),
     "linhas_silver": spark.table(f"{DESTINO}.{s}").count()}
    for b, s in PARES])
conservacao["diferenca"] = conservacao["linhas_bronze"] - conservacao["linhas_silver"]
display(conservacao)

## Validação 2 — As regras foram aplicadas?

Cada coluna abaixo deve retornar **zero** (ou, no caso da última, o número de pedidos utilizáveis nas métricas de prazo). Esta célula é a evidência "depois" que se compara com o diagnóstico "antes" do MVP05.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM mvp.silver.avaliacoes)              AS avaliacoes_duplicadas_por_pedido,
  (SELECT COUNT(*) FROM mvp.silver.avaliacoes WHERE nota NOT BETWEEN 1 AND 5)           AS notas_fora_do_dominio,
  (SELECT COUNT(*) FROM mvp.silver.pagamentos WHERE parcelas < 1)                       AS parcelas_invalidas,
  (SELECT COUNT(*) FROM mvp.silver.produtos   WHERE categoria IS NULL)                  AS produtos_sem_categoria,
  (SELECT COUNT(*) FROM mvp.silver.produtos   WHERE categoria_en IS NULL)               AS produtos_sem_traducao,
  (SELECT COUNT(*) FROM mvp.silver.produtos   WHERE peso_g <= 0)                        AS pesos_invalidos,
  (SELECT COUNT(*) FROM mvp.silver.clientes   WHERE length(cep_prefixo) <> 5)           AS ceps_fora_do_padrao,
  (SELECT COUNT(*) FROM mvp.silver.pedidos    WHERE data_hora_compra IS NULL)           AS pedidos_sem_data_compra,
  (SELECT SUM(CASE WHEN flag_entrega_valida THEN 1 ELSE 0 END) FROM mvp.silver.pedidos) AS pedidos_com_entrega_valida;

## Validação 3 — Efeito da padronização de cidades

In [0]:
%sql
SELECT 'clientes' AS tabela,
       (SELECT COUNT(DISTINCT customer_city) FROM mvp.bronce.customers) AS cidades_bronze,
       (SELECT COUNT(DISTINCT cidade)        FROM mvp.silver.clientes)  AS cidades_silver
UNION ALL
SELECT 'vendedores',
       (SELECT COUNT(DISTINCT seller_city) FROM mvp.bronce.sellers),
       (SELECT COUNT(DISTINCT cidade)      FROM mvp.silver.vendedores);

## Validação 4 — Tipagem aplicada

Diferente da Bronze (tudo STRING), a Silver deve exibir TIMESTAMP, INT, DECIMAL e BOOLEAN.

In [0]:
spark.table("mvp.silver.pedidos").printSchema()
spark.table("mvp.silver.itens_pedido").printSchema()

## Resultado

Sete tabelas Silver persistidas em `mvp.silver`, tipadas, padronizadas e com cada regra rastreável a um achado do MVP05. Próximo passo: **MVP07-gold**, que materializa o modelo dimensional desenhado no MVP03.